## Flat project v4 note
This project bundle has been flattened. The API module is now `api.py` and the recommended run command is `uvicorn api:app --reload`.

# Finnoobs — XchangeFinder Data Exploration

This notebook explores the SQLite database shipped with the original **Finnoobs / XchangeFinder** project (Team 17, SocGen Finhack 2017).

**Scope:** database inspection, data quality checks, store safety ratings, user/store exchange-rate submissions, and visual summaries. It does not claim to train a machine-learning model or to provide live foreign-exchange rates.

> Run from the repository root or from the `finnoobs-master` directory. The notebook locates the bundled database automatically.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
candidates = [
    Path("db/XchangeFinder.db"),
    Path("finnoobs-master/db/XchangeFinder.db"),
]
DB_PATH = next((p for p in candidates if p.exists()), None)
if DB_PATH is None:
    raise FileNotFoundError("Could not find db/XchangeFinder.db. Run the notebook from the repository root or project folder.")
print(f"Using database: {DB_PATH.resolve()}")

## 1. Database structure

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query(
        "SELECT name, sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
        conn
    )
display(tables[["name"]])
for _, row in tables.iterrows():
    print(f"\\n### {row['name']}\\n{row['sql']}")

## 2. Load tables

In [ ]:
TABLE_NAMES = ["stores", "users", "safetyRatings", "storeRateSubmissions", "userRateSubmissions"]
data = {}
with sqlite3.connect(DB_PATH) as conn:
    for table in TABLE_NAMES:
        try:
            data[table] = pd.read_sql_query(f'SELECT * FROM "{table}"', conn)
            print(f"{table}: {len(data[table]):,} rows")
        except Exception as exc:
            print(f"{table}: not loaded ({exc})")

stores = data.get("stores", pd.DataFrame())
ratings = data.get("safetyRatings", pd.DataFrame())
store_rates = data.get("storeRateSubmissions", pd.DataFrame())
user_rates = data.get("userRateSubmissions", pd.DataFrame())
# Hide credential columns in previews; the legacy database contains plaintext credentials.
safe_stores = stores.drop(columns=["password", "username"], errors="ignore")
display(safe_stores.head())


## 3. Data quality overview

In [ ]:
quality_rows = []
for name, df in data.items():
    quality_rows.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()) if not df.empty else 0,
        "missing_cells": int(df.isna().sum().sum()) if not df.empty else 0,
    })
quality = pd.DataFrame(quality_rows)
display(quality)


## 4. Store safety ratings

In [ ]:
if not ratings.empty and {"storeUUID", "rating"}.issubset(ratings.columns):
    rating_summary = ratings.groupby("storeUUID").agg(
        rating_count=("rating", "count"),
        average_rating=("rating", "mean"),
    ).reset_index()
    display(rating_summary.sort_values(["average_rating", "rating_count"], ascending=False).head(15))
    ax = rating_summary.sort_values("average_rating").plot(
        x="storeUUID", y="average_rating", kind="bar", figsize=(10, 4), legend=False,
        title="Average safety rating by store"
    )
    ax.set_xlabel("Store ID")
    ax.set_ylabel("Average rating")
    plt.tight_layout()
    plt.show()
else:
    print("No usable safety rating rows/columns were found in this database.")


## 5. Exchange-rate submissions

In [ ]:
def summarize_rates(df, source_name):
    required = {"fromCurrency", "toCurrency", "rate"}
    if df.empty or not required.issubset(df.columns):
        print(f"{source_name}: no usable exchange-rate records")
        return pd.DataFrame()
    out = df.copy()
    out["rate"] = pd.to_numeric(out["rate"], errors="coerce")
    summary = (out.dropna(subset=["rate"])
        .groupby(["fromCurrency", "toCurrency"])
        .agg(submissions=("rate", "size"), mean_rate=("rate", "mean"),
             median_rate=("rate", "median"), min_rate=("rate", "min"), max_rate=("rate", "max"))
        .reset_index())
    summary.insert(0, "source", source_name)
    return summary

rate_summary = pd.concat([
    summarize_rates(store_rates, "store"),
    summarize_rates(user_rates, "user"),
], ignore_index=True)
display(rate_summary)


## 6. Submission volume by currency pair

In [ ]:
if not rate_summary.empty:
    volume = rate_summary.pivot_table(
        index=["fromCurrency", "toCurrency"], columns="source",
        values="submissions", aggfunc="sum", fill_value=0
    )
    display(volume)
    ax = volume.plot(kind="bar", figsize=(11, 5), title="Exchange-rate submission counts by currency pair")
    ax.set_xlabel("Currency pair")
    ax.set_ylabel("Number of submissions")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No exchange-rate submissions to visualize.")


## 10. Interpretation and limitations

- The bundled SQLite file is a historical project dataset; it should not be treated as current market data.
- A submitted rate is a user/store report, not a verified bank or market quote.
- The original web application contains legacy code and security weaknesses (including plaintext password handling and disabled CSRF protection). Do not deploy it publicly without a security-focused rewrite.
- The notebook is an exploratory analysis layer. It does not modify the source database.
- Missing location enrichment or external API credentials can affect the old Flask app; the analysis here does not require geocoding or network access.

## 11. Suggested next development steps

1. Replace plaintext passwords with Werkzeug password hashing.
2. Enable CSRF protection and use environment-based secret keys.
3. Parameterize every SQL query and validate store identifiers.
4. Add automated tests for database access, authentication, and currency-pair filtering.
5. Separate the Flask application from the analysis notebook and document setup instructions.

# 7. Generated 2D and 3D diagrams

The following diagrams are generated from the project structure and bundled historical dataset. They are also saved under `docs/diagrams/` for use in the GitHub README.

**2D diagrams**
- System architecture: browser, Flask application, SQLite and notebook analytics.
- Database relationship overview: stores, users, ratings and rate submissions.

**3D diagrams**
- Layered architecture view: conceptual presentation, application, data and analytics layers.
- Exchange-rate scatter: historical submitted rates by currency pair and source. The vertical axis is a record index, not time or a prediction.

The architecture views are conceptual documentation, while the exchange-rate chart is based on rows in the bundled database.

In [ ]:
from IPython.display import display, Image
from pathlib import Path

diagram_dir = Path("docs/diagrams")
if not diagram_dir.exists():
    diagram_dir = Path("../docs/diagrams")

diagram_files = [
    ("System architecture — 2D", "architecture_2d.png"),
    ("Database relationships — 2D", "database_relationships_2d.png"),
    ("Layered architecture — 3D", "layered_architecture_3d.png"),
    ("Historical exchange-rate submissions — 3D", "exchange_rates_3d.png"),
]
for caption, filename in diagram_files:
    path = diagram_dir / filename
    print(caption)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"Diagram not found: {path}. Run the repository diagram-generation script or open the notebook from the repository root.")


# 8. Comparison table: store vs user rate submissions

This comparison uses the original database rows. A submission is a reported value, not an independently verified quote. Currency pairs can have different scales, so compare rates only within the same pair.

In [ ]:
def source_comparison(store_df, user_df):
    frames = []
    for source, frame in [("Store submissions", store_df), ("User submissions", user_df)]:
        if frame.empty or not {"fromCurrency", "toCurrency", "rate"}.issubset(frame.columns):
            continue
        tmp = frame.copy()
        tmp["rate"] = pd.to_numeric(tmp["rate"], errors="coerce")
        tmp = tmp.dropna(subset=["rate"])
        grouped = tmp.groupby(["fromCurrency", "toCurrency"])["rate"].agg(
            submission_count="count", mean_rate="mean", median_rate="median",
            minimum_rate="min", maximum_rate="max"
        ).reset_index()
        grouped.insert(0, "source", source)
        frames.append(grouped)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

comparison_table = source_comparison(store_rates, user_rates)
display(comparison_table.round(4))

# Compact overall comparison, including counts and coverage.
overall = pd.DataFrame([
    {
        "source": "Store submissions",
        "records": len(store_rates),
        "currency_pairs": store_rates[["fromCurrency", "toCurrency"]].drop_duplicates().shape[0]
            if not store_rates.empty else 0,
        "linked_stores": store_rates["storeUUID"].nunique() if "storeUUID" in store_rates else 0,
    },
    {
        "source": "User submissions",
        "records": len(user_rates),
        "currency_pairs": user_rates[["fromCurrency", "toCurrency"]].drop_duplicates().shape[0]
            if not user_rates.empty else 0,
        "linked_stores": user_rates["storeUUID"].nunique() if "storeUUID" in user_rates else 0,
    },
])
display(overall)


## 9. Comparison interpretation

- **Store submissions** are rates entered against a store identifier.
- **User submissions** are rates entered by a user and associated with a store identifier.
- The table compares record counts and descriptive statistics by currency pair.
- A larger number of submissions does not prove a rate is more accurate.
- The dataset timestamps are from 2017; do not use these values for current currency conversion.

# 12. FastAPI analytics service (upgrade)

A separate read-only FastAPI service provides a modern REST interface and interactive Swagger/OpenAPI documentation. Start it from the project root with `uvicorn app.main:app --reload`. Open `/docs` for Swagger UI and `/redoc` for ReDoc.

Endpoints:
- `GET /health` — service/database availability
- `GET /tables` — table listing
- `GET /rates?source=all&from_currency=USD&to_currency=EUR&limit=500` — filtered records
- `GET /analytics/pairs` — pair-wise statistics across store and user submissions
- `GET /analytics/export.csv?source=all` — downloadable CSV export

Set `XCHANGEFINDER_DB` to the database path if it is not at `db/XchangeFinder.db`. Database connections use SQLite read-only URI mode; filter values are bound parameters. This service does not return password columns.


In [ ]:
# API launch command (run in a terminal, not in the notebook kernel):
# pip install -r requirements.txt
# uvicorn app.main:app --reload
# Swagger: http://127.0.0.1:8000/docs


## 13. API analytics workflow

Use `/analytics/pairs` for source-level currency-pair statistics and `/analytics/export.csv` for offline analysis. These statistics describe historical submissions in the bundled dataset, not live market rates. Compare only like-for-like currency pairs; different pairs have different scales.


## 14. Security and test notes

- The API opens the SQLite database in read-only mode.
- Query filter values are passed as SQL parameters.
- Table identifiers used in SQL are selected from a fixed internal whitelist.
- Password and credential fields are not exposed by API responses.
- CI runs basic endpoint tests with GitHub Actions. Tests are smoke tests and do not substitute for additional integration/security tests.
- Do not publish this historical dataset or original application without reviewing licensing, data sensitivity, and deployment security.


## Upgrade v2: API analytics additions

The v2 service adds pagination, pair summaries, source comparison, and two CSV export routes. This dataset contains submitted historical rates and should not be treated as live exchange-rate quotes.


### REST endpoints
- `GET /rates?source=all&limit=200&offset=0`
- `GET /analytics/pairs`
- `GET /analytics/compare?from_currency=USD&to_currency=EUR`
- `GET /analytics/export.csv`
- `GET /analytics/pairs.csv`
- Swagger: `/docs`


In [ ]:
# Example: retrieve grouped historical comparison using the API
import requests
BASE_URL = "http://127.0.0.1:8000"
response = requests.get(f"{BASE_URL}/analytics/compare", params={"from_currency":"USD", "to_currency":"EUR"}, timeout=15)
response.raise_for_status()
response.json()


### CSV exports
Download raw submissions from `/analytics/export.csv` or aggregated pair statistics from `/analytics/pairs.csv`. Filtered output can be used in spreadsheet tools or further notebook analysis.

### Security notes
SQLite is opened with read-only URI mode and query-only pragma. SQL values use bound parameters; internal table identifiers are allowlisted. Do not expose this API publicly without authentication, HTTPS, rate limits and deployment review.


## v3: Data Science, Machine Learning and Language Integration

New endpoints: `/analytics/data-quality` and experimental `/analytics/ml/forecast`. This notebook section documents Data Analyst and ML workflows and the language starter tracks (Python, C, C++, Go, R, Rust and F#). Deep learning (LSTM/GRU) is a future optional experiment requiring timestamps, sufficient observations, chronological validation and careful evaluation.

In [ ]:
# Experiment with the ML API (start the API server first)
import requests
base = "http://127.0.0.1:8000"
quality = requests.get(f"{base}/analytics/data-quality", timeout=10).json()
print(quality)
# Example: update currencies to a pair present in your database
# forecast = requests.get(f"{base}/analytics/ml/forecast", params={"from_currency":"USD", "to_currency":"EUR", "steps":3}, timeout=10)
# print(forecast.json())


# XchangeFinder Visualization & Analytics Lab

This notebook integrates comparison tables, charts, and 3D visualizations into the project.

**Important:** The generated demo data is synthetic. Replace it with API-returned historical submission data for real analysis. Visualizations do not imply future rates or investment advice.

## 1. Load API data or create reproducible demo data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # registers 3D plotting
from IPython.display import display

# Synthetic demo data, replace with API data later
rng = np.random.default_rng(42)
pairs = ['USD/EUR', 'USD/INR', 'EUR/INR', 'GBP/USD', 'JPY/USD']
demo = pd.DataFrame({
    'pair': pairs,
    'store_mean': [0.91, 83.2, 91.3, 1.27, 0.0067],
    'user_mean':  [0.915, 83.5, 91.0, 1.265, 0.0068],
    'submissions': [124, 98, 80, 61, 45],
})
demo['difference'] = demo['user_mean'] - demo['store_mean']
demo['relative_difference_pct'] = demo['difference'] / demo['store_mean'] * 100
display(demo)


## 2. Pair comparison table — summary and relative difference

In [ ]:
display(demo.style.format({
    'store_mean':'{:.5f}', 'user_mean':'{:.5f}',
    'difference':'{:+.5f}', 'relative_difference_pct':'{:+.2f}%'
}).background_gradient(subset=['relative_difference_pct'], cmap='RdYlGn'))

## 3. Grouped comparison chart

In [ ]:
x = np.arange(len(demo))
width = 0.36
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x-width/2, demo['store_mean'], width, label='Store average')
ax.bar(x+width/2, demo['user_mean'], width, label='User average')
ax.set_xticks(x, demo['pair'])
ax.set_title('Store vs User submitted averages (demo)')
ax.set_ylabel('Reported rate')
ax.legend()
ax.grid(axis='y', alpha=.25)
plt.tight_layout()
plt.show()

## 4. Difference chart (absolute and relative)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(demo['pair'], demo['difference'])
axes[0].set_title('Absolute difference: User − Store')
axes[0].tick_params(axis='x', rotation=30)
axes[1].bar(demo['pair'], demo['relative_difference_pct'])
axes[1].set_title('Relative difference (%)')
axes[1].tick_params(axis='x', rotation=30)
for ax in axes: ax.grid(axis='y', alpha=.25)
plt.tight_layout()
plt.show()

## 5. Submission volume chart

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(demo['pair'], demo['submissions'])
ax.set_title('Submission volume by currency pair (demo)')
ax.set_xlabel('Number of submissions')
ax.grid(axis='x', alpha=.25)
plt.tight_layout()
plt.show()

## 6. 3D visualization — rate comparison and submission volume

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
xs = np.arange(len(demo))
ax.bar3d(xs-.15, np.zeros(len(demo)), np.zeros(len(demo)),
          .3, np.ones(len(demo)), demo['store_mean'], alpha=.8)
ax.bar3d(xs+.15, np.ones(len(demo)), np.zeros(len(demo)),
          .3, np.ones(len(demo)), demo['user_mean'], alpha=.8)
ax.set_xticks(xs, demo['pair'])
ax.set_yticks([0.5, 1.5], ['Store', 'User'])
ax.set_zlabel('Average submitted rate')
ax.set_title('3D rate comparison by currency pair (demo)')
plt.tight_layout()
plt.show()

## 7. 3D scatter — submissions, difference, and volume

In [ ]:
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(demo['store_mean'], demo['user_mean'], demo['submissions'], s=70)
for i, row in demo.iterrows():
    ax.text(row['store_mean'], row['user_mean'], row['submissions'], row['pair'])
ax.set_xlabel('Store average')
ax.set_ylabel('User average')
ax.set_zlabel('Submission count')
ax.set_title('3D scatter: rate comparison and submission volume (demo)')
plt.tight_layout()
plt.show()

## 8. Export notebook summary to CSV

In [ ]:
demo.to_csv('currency_pair_comparison_demo.csv', index=False)
print('CSV saved: currency_pair_comparison_demo.csv')

## Using actual API data

Example approach:
```python
import requests
response = requests.get('http://127.0.0.1:8000/analytics/pairs')
response.raise_for_status()
api_pairs = pd.DataFrame(response.json()['pairs'])
display(api_pairs.head())
```
The API's current summary schema and your database contents may vary. Inspect returned columns before plotting. Demo charts above use synthetic values.

# Interactive Analytics Upgrade — Plotly, Heatmaps, Time Series & Model Evaluation

This section uses **synthetic illustrative data** unless actual database/API observations are loaded. Forecast model comparisons are an example backtest on synthetic sequential observations and must not be interpreted as predictive financial performance.

In [ ]:
# Install/requirements now include Plotly. Restart the kernel if newly installed.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit


## 9. Interactive Plotly 3D chart

In [ ]:
# Interactive 3D comparison (using demo table defined earlier)
fig = px.scatter_3d(demo, x='store_mean', y='user_mean', z='submissions',
                    color='pair', size='submissions', hover_name='pair',
                    title='Interactive 3D — Store vs User averages and submission count')
fig.show()


## 10. Correlation heatmap

In [ ]:
numeric_demo = demo[['store_mean','user_mean','submissions','difference','relative_difference_pct']]
corr = numeric_demo.corr(numeric_only=True)
fig = px.imshow(corr, text_auto='.2f', aspect='auto', color_continuous_scale='RdBu_r',
                title='Demo data correlation matrix')
fig.show()
display(corr.round(3))


## 11. Time-series analytics example

In [ ]:
# Synthetic timestamped demo values; replace with actual dated submissions.
ts_rng = np.random.default_rng(2026)
dates = pd.date_range('2025-01-01', periods=180, freq='D')
synthetic_series = pd.DataFrame({
    'date': dates,
    'rate': 1.08 + np.linspace(0, .025, len(dates)) + ts_rng.normal(0, .008, len(dates))
})
synthetic_series['rolling_mean_14'] = synthetic_series['rate'].rolling(14).mean()
fig = px.line(synthetic_series, x='date', y=['rate','rolling_mean_14'],
              title='Synthetic daily rate and 14-day rolling average')
fig.show()


## 12. Chronological model accuracy comparison (demo)

In [ ]:
# Demonstration regression model comparison with chronological split.
# Synthetic series: simple lag features, not financial-grade validation.
model_data = synthetic_series.copy()
model_data['lag1'] = model_data['rate'].shift(1)
model_data['lag7'] = model_data['rate'].shift(7)
model_data = model_data.dropna().reset_index(drop=True)
X = model_data[['lag1','lag7']]
y = model_data['rate']
split = int(len(model_data)*.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
models = {'Linear Regression': LinearRegression(), 'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)}
scores=[]
for name, model in models.items():
    model.fit(X_train, y_train)
    preds=model.predict(X_test)
    scores.append({'model':name,'MAE':mean_absolute_error(y_test,preds),
                   'RMSE':mean_squared_error(y_test,preds)**0.5})
score_df=pd.DataFrame(scores)
display(score_df)
fig=px.bar(score_df, x='model', y=['MAE','RMSE'], barmode='group',
           title='Demo model error comparison (lower is better)')
fig.show()


## 13. Automated HTML dashboard

In [ ]:
# Build a small shareable interactive HTML dashboard from demo analytics.
dashboard_figs=[]
dashboard_figs.append(px.bar(demo, x='pair', y=['store_mean','user_mean'], barmode='group',
                             title='Store vs User averages'))
dashboard_figs.append(px.imshow(corr, text_auto='.2f', aspect='auto',
                               title='Correlation heatmap'))
dashboard_figs.append(px.line(synthetic_series, x='date', y=['rate','rolling_mean_14'],
                             title='Time series and rolling mean'))
dashboard_figs.append(px.bar(score_df, x='model', y=['MAE','RMSE'], barmode='group',
                             title='Model accuracy comparison'))
html_parts=['<h1>XchangeFinder Interactive Analytics Dashboard</h1>',
            '<p>Illustrative synthetic data. Replace with real historical data before analysis.</p>']
for f in dashboard_figs:
    html_parts.append(f.to_html(full_html=False, include_plotlyjs='cdn'))
with open('XchangeFinder_analytics_dashboard.html','w',encoding='utf-8') as f:
    f.write('<html><head><meta charset="utf-8"></head><body>'+''.join(html_parts)+'</body></html>')
print('Dashboard generated: XchangeFinder_analytics_dashboard.html')


# 15. Visualization reference gallery

The following cropped figures were supplied as visual references and are included with this repository.
They demonstrate architecture, comparative charts, 3D scatter, classification, signal-intensity and
parametric visualizations. They are **reference images**, not notebook-generated results and not
measurements from the bundled historical exchange-rate database.

For reproducible XchangeFinder analysis, use the notebook's API-backed or clearly labelled synthetic
currency-pair examples. In particular, the Iris/species and signal-intensity figures are unrelated
sample visualization domains and must not be interpreted as FX data.

![System architecture](docs/images/01-system-architecture.png)\n\n![Grouped comparison chart](docs/images/02-comparison-bar-chart.png)\n\n![3D signal intensity bar chart](docs/images/03-3d-signal-intensity.png)\n\n![Iris 3D scatter plot](docs/images/04-iris-3d-scatter.png)\n\n![3D clustered scatter plot](docs/images/05-3d-cluster-scatter.png)\n\n![3D time-series visualization](docs/images/06-3d-time-series.png)\n\n![3D classification scatter with tooltip](docs/images/07-3d-classification-tooltip.png)\n\n![3D parametric spiral](docs/images/08-3d-parametric-spiral.png)\n\n![3D Iris classification with tooltip](docs/images/09-iris-classification-tooltip.png)\n\n